
# Polarization curve

Assemble a PEM fuel cell from first principles and compute a steady-state
polarization curve with a single vectorised call.

``marapendi`` separates the *description* of the cell (a tree of dataclasses
holding geometry and material parameters) from the *physics* (model objects
that receive the cell and operating conditions at solve time).  This example
walks through both steps and plots the resulting V–i curve, power density,
high-frequency resistance, and individual loss contributions.


## Cell assembly

A :class:`~marapendi.cell.fuelcell.FuelCell` is built from nested dataclasses.
The anode and cathode each own a catalyst layer, a gas diffusion layer, and a
flow channel.  A :class:`~marapendi.membrane.pem.PFSA` membrane closes the MEA.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import marapendi as mrpd

liq = mrpd.DarcyTransportModel(J_function_exponent=2)
ionomer = mrpd.PFSAIonomer(equivalent_weight=1100, dry_density=1980)

ca_cl = mrpd.PtCCatalystLayer(
    ecsa=70e3,
    platinum_loading=0.4e-2,
    ionomer=ionomer,
    reaction=mrpd.ElectrochemicalReaction(
        reference_exchange_current_density=2.5e-4,
        reaction_order=0.54,
        activation_energy=67e6,
        reference_activity=1e5,
        reference_temperature=353.15,
        number_of_electrons=2,
        charge_transfer_coeff=0.5,
    ),
    thickness=10e-6,
    thermal_conductivity=0.22,
    pore_diameter=40e-9,
    absolute_permeability=1e-13,
    contact_angle=97.0,
    two_phase_transport_model=liq,
)

gdl = mrpd.GasDiffusionLayer(
    thickness=200e-6,
    porosity=0.6,
    contact_angle=120.0,
    effective_gas_diffusion_ratio=0.3,
    absolute_permeability=1e-12,
    thermal_conductivity=0.5,
    two_phase_transport_model=liq,
)

cell = mrpd.FuelCell(
    area=25e-4,
    electrical_resistance=30e-7,
    ca=mrpd.FuelCellSide(
        cl=ca_cl,
        gdl=gdl,
        ch=mrpd.FlowChannel(
            width=1e-3, height=1e-3, length=0.1, n_parallel=20, reactant="o2"
        ),
        has_mpl=False,
        thermal_contact_resistance=4e-4,
    ),
    an=mrpd.FuelCellSide(
        cl=mrpd.PtCCatalystLayer(thickness=5e-6, two_phase_transport_model=liq),
        gdl=mrpd.GasDiffusionLayer(
            thickness=200e-6,
            effective_gas_diffusion_ratio=0.3,
            thermal_conductivity=0.5,
            two_phase_transport_model=liq,
        ),
        ch=mrpd.FlowChannel(
            width=1e-3, height=1e-3, length=0.1, n_parallel=20, reactant="h2"
        ),
        has_mpl=False,
        thermal_contact_resistance=4e-4,
    ),
    membrane=mrpd.PFSA(ionomer=ionomer, dry_thickness=25e-6),
)

## Operating conditions and vectorised solve

:class:`~marapendi.simulation.conditions.CellConditions` accepts numpy arrays
for ``current_density`` — all operating points are evaluated in one call.



In [ ]:
T = 353.15  # K  (80 °C)
i_arr = np.linspace(200, 22000, 60)  # A/m²

conditions = mrpd.CellConditions(
    current_density=i_arr,
    cell_temperature=T,
    ca=mrpd.SideConditions(
        inlet_temperature=T,
        outlet_pressure=1.5e5,
        dry_o2_mole_fraction=0.21,
        inlet_relative_humidity=0.5,
        stoichiometry=2.0,
    ),
    an=mrpd.SideConditions(
        inlet_temperature=T,
        outlet_pressure=1.5e5,
        dry_h2_mole_fraction=1.0,
        inlet_relative_humidity=0.5,
        stoichiometry=1.5,
    ),
)

model = mrpd.ExplicitSteadyStateModel()
state = model.set_initial_conditions(cell, conditions)
state = model.solve(cell, conditions, state)

## Polarization and power-density curves



In [ ]:
i_cm2 = i_arr * 1e-4  # A/cm²
V = state.cell_voltage
P = V * i_cm2          # W/cm²

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.8))

ax1.plot(i_cm2, V, "o-", ms=4, color="C0")
ax1.set_xlabel("Current density (A cm⁻²)")
ax1.set_ylabel("Cell voltage (V)")
ax1.set_ylim(0, 1.1)
ax1.grid(True, alpha=0.3)
ax1.set_title("Polarization curve")

ax2.plot(i_cm2, P, "s-", ms=4, color="C1")
ax2.set_xlabel("Current density (A cm⁻²)")
ax2.set_ylabel("Power density (W cm⁻²)")
ax2.grid(True, alpha=0.3)
ax2.set_title("Power density")

fig.tight_layout()

## High-frequency resistance

HFR is the total ohmic resistance (membrane proton resistance +
electrical contact resistance), in Ω·m².



In [ ]:
hfr = model.voltage_model.high_frequency_resistance(cell, state)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(i_cm2, hfr * 1e4, "D-", ms=4, color="C2")
ax.set_xlabel("Current density (A cm⁻²)")
ax.set_ylabel("HFR (mΩ cm²)")
ax.set_title("High-frequency resistance")
ax.grid(True, alpha=0.3)
fig.tight_layout()

## Loss breakdown

``CellState.E_rev``, ``eta_act``, and ``eta_ohm`` give the voltage balance:
V_cell = E_rev − η_act − η_ohm.



In [ ]:
V_rev   = state.E_rev
eta_act = state.eta_act
eta_ohm = state.eta_ohm

fig, ax = plt.subplots(figsize=(7, 4))
ax.stackplot(
    i_cm2,
    [np.abs(eta_act), np.abs(eta_ohm)],
    labels=["Activation (η_act)", "Ohmic (η_ohm)"],
    colors=["C3", "C2"],
    alpha=0.75,
)
ax.plot(i_cm2, V_rev - V, "k--", lw=1.5, label="Total loss")
ax.set_xlabel("Current density (A cm⁻²)")
ax.set_ylabel("Overpotential (V)")
ax.set_title("Loss breakdown")
ax.legend(loc="upper left", fontsize=9)
ax.grid(True, alpha=0.3)
fig.tight_layout()